# Learning 01: Gradio Basics

Gradio lets you build interactive ML demos in pure Python.

Two main building blocks:
- **`gr.Interface`** — wrap a function with inputs/outputs automatically
- **`gr.Blocks`** — full control over layout, events, and state

```
gr.Interface(fn, inputs, outputs)   ← simple, one function
gr.Blocks()                         ← flexible, multi-component
```

In [ ]:
import gradio as gr
print(f"Gradio version: {gr.__version__}")

## 1. gr.Interface — Simplest Form

In [ ]:
def reverse_text(text: str) -> str:
    return text[::-1]


demo = gr.Interface(
    fn=reverse_text,
    inputs=gr.Textbox(label="Input text", placeholder="Type something..."),
    outputs=gr.Textbox(label="Reversed"),
    title="Text Reverser",
    description="Reverses any text you type.",
    examples=[["Hello world"], ["Gradio is great"]],
)

demo.launch()

## 2. Key Input/Output Components

| Component | Use case |
|-----------|----------|
| `gr.Textbox` | Text input/output |
| `gr.Dropdown` | Select from options |
| `gr.CheckboxGroup` | Multi-select |
| `gr.Slider` | Numeric range |
| `gr.Label` | Classification result with probabilities |
| `gr.HighlightedText` | Text with colored entity spans |
| `gr.JSON` | Raw JSON output |
| `gr.Markdown` | Rich text display |

In [ ]:
# gr.Label — classification probabilities
def classify_mock(text: str) -> dict:
    """Mock classifier — returns label → probability dict."""
    words = text.lower().split()
    scores = {
        "positive": sum(w in {"good", "great", "love", "excellent"} for w in words) * 0.3 + 0.1,
        "negative": sum(w in {"bad", "terrible", "hate", "awful"} for w in words) * 0.3 + 0.1,
        "neutral": 0.5,
    }
    total = sum(scores.values())
    return {k: round(v / total, 3) for k, v in scores.items()}


demo = gr.Interface(
    fn=classify_mock,
    inputs=gr.Textbox(label="Text"),
    outputs=gr.Label(num_top_classes=3, label="Sentiment"),
    examples=[["This is great!"], ["Terrible experience"]],
)
demo.launch()

In [ ]:
# gr.HighlightedText — NER-style entity spans
# Format: list of {"entity": label, "start": int, "end": int}
# OR: list of (text_chunk, label_or_None) tuples

def highlight_mock(text: str) -> list:
    """Returns (chunk, label) pairs for HighlightedText."""
    # Simple: highlight any word longer than 6 chars
    result = []
    for word in text.split():
        if len(word) > 6:
            result.append((word + " ", "LONG"))
        else:
            result.append((word + " ", None))
    return result


demo = gr.Interface(
    fn=highlight_mock,
    inputs=gr.Textbox(label="Text", value="Gradio makes building demos extremely simple"),
    outputs=gr.HighlightedText(label="Highlighted", color_map={"LONG": "#ff9999"}),
)
demo.launch()

## 3. gr.Blocks — Custom Layout

In [ ]:
with gr.Blocks(title="My App") as demo:
    gr.Markdown("## Text Tools")

    with gr.Row():
        text_in = gr.Textbox(label="Input", lines=3)
        text_out = gr.Textbox(label="Output", lines=3)

    with gr.Row():
        btn_upper = gr.Button("UPPERCASE")
        btn_lower = gr.Button("lowercase")
        btn_count = gr.Button("Word count")

    # Wire buttons to functions
    btn_upper.click(fn=str.upper, inputs=text_in, outputs=text_out)
    btn_lower.click(fn=str.lower, inputs=text_in, outputs=text_out)
    btn_count.click(
        fn=lambda t: f"{len(t.split())} words",
        inputs=text_in,
        outputs=text_out
    )

demo.launch()

## 4. gr.Tabs — Multi-page Layout

In [ ]:
with gr.Blocks() as demo:
    gr.Markdown("# NLP Toolkit")

    with gr.Tabs():
        with gr.Tab("Reverse"):
            t1_in = gr.Textbox(label="Text")
            t1_out = gr.Textbox(label="Reversed")
            gr.Button("Run").click(lambda t: t[::-1], t1_in, t1_out)

        with gr.Tab("Word Count"):
            t2_in = gr.Textbox(label="Text", lines=5)
            t2_out = gr.JSON(label="Stats")
            def word_stats(text):
                words = text.split()
                return {"words": len(words), "chars": len(text), "sentences": text.count('.')}
            gr.Button("Analyze").click(word_stats, t2_in, t2_out)

demo.launch()

## 5. gr.State — Persistent State Between Calls

In [ ]:
with gr.Blocks() as demo:
    gr.Markdown("## Click Counter")
    count_state = gr.State(value=0)      # persists per session
    count_display = gr.Textbox(label="Count", value="0")
    btn = gr.Button("Click me")

    def increment(count):
        count += 1
        return count, str(count)  # new state, display value

    btn.click(
        fn=increment,
        inputs=count_state,
        outputs=[count_state, count_display]
    )

demo.launch()